# Bengaluru House Price Prediction
## Session 1: setup, load and first-pass cleaning

**Objective.** Load the raw Bengaluru house-price data, understand it, and reduce it to a clean frame that holds only the columns that can feed the model (`location`, `size`, `total_sqft`, `bath`, `price`) with no missing values.

**Target.** `price`, in **lakhs of INR** (1 lakh = 100,000 INR). Prices stay in lakhs throughout the project; nothing is converted here.

**Out of scope for this session.** Cleaning `total_sqft` (ranges and unit strings), encoding, scaling and the train/test split.

## 1. Setup and load

`df_raw` holds the data exactly as loaded and is never modified; every cleaning step works on a new frame.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")

In [ ]:
CSV_NAME = "bengaluru_house_prices.csv"

# The notebook lives in model/, so the data folder is one level up.
# "data/" also works when Jupyter is started from the repository root.
candidates = [Path("../data") / CSV_NAME, Path("data") / CSV_NAME]
csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        f"{CSV_NAME} not found. Download 'Bengaluru House Price Data' from Kaggle "
        f"and save it as data/{CSV_NAME} in the repository root."
    )

df_raw = pd.read_csv(csv_path)
print(f"Loaded {csv_path}")

## 2. Inspect the raw data

In [ ]:
df_raw.shape

In [ ]:
df_raw.head()

In [ ]:
df_raw.dtypes

Text columns appear as `object` in pandas below 3 and as `str` from pandas 3 on; either way they are non-numeric. `total_sqft` is one of them: it holds ranges and unit strings, so it stays untouched until Session 2.

In [ ]:
null_counts = df_raw.isnull().sum()
pd.DataFrame({"nulls": null_counts, "pct_missing": (null_counts / len(df_raw) * 100).round(2)})

In [ ]:
df_raw["area_type"].value_counts()

### Observed vs expected

The brief quotes approximate figures. The table puts them next to what the data contains.

In [ ]:
expected = {  # figures quoted in the Session 1 brief (all approximate)
    "rows": 13000,
    "columns": 9,
    "society missing (%)": 40,
    "balcony missing": 600,
    "bath missing": 70,
    "size missing": 16,
    "location missing": 1,
}
observed = {
    "rows": len(df_raw),
    "columns": df_raw.shape[1],
    "society missing (%)": round(df_raw["society"].isnull().mean() * 100, 1),
    "balcony missing": int(df_raw["balcony"].isnull().sum()),
    "bath missing": int(df_raw["bath"].isnull().sum()),
    "size missing": int(df_raw["size"].isnull().sum()),
    "location missing": int(df_raw["location"].isnull().sum()),
}
pd.DataFrame({
    "expected (approx.)": pd.Series(expected, dtype=object),
    "observed": pd.Series(observed, dtype=object),
})

## 3. Drop columns that cannot feed the model

Evidence for each decision, computed from the raw data:

In [ ]:
DROP_COLS = ["area_type", "availability", "society", "balcony"]

evidence = pd.DataFrame({
    "dtype": df_raw[DROP_COLS].dtypes.astype(str),
    "pct_missing": (df_raw[DROP_COLS].isnull().mean() * 100).round(1),
    "n_unique": df_raw[DROP_COLS].nunique(),
    "top_values": pd.Series({c: df_raw[c].value_counts().head(3).to_dict() for c in DROP_COLS}),
})
with pd.option_context("display.max_colwidth", None):
    display(evidence)

**Justification for each drop**

- **`area_type`**: it changes what `total_sqft` measures (super built-up, built-up, carpet or plot area), so it may well carry signal; it is left out of this first baseline as a judgment call, not because it is unusable, and it is logged in the README as a known limitation to test in Session 5.
- **`availability`**: it mixes the label "Ready To Move" with possession-date strings (see the table above), so it describes when a unit can be occupied rather than what the property is, and it cannot be used without parsing dates.
- **`society`**: a large share of its values are missing and it has a very large number of distinct names (see the table above), so it would be a sparse, high-cardinality categorical that cannot be encoded reliably.
- **`balcony`**: it is a minor amenity count with a small range of values and a few hundred missing entries (see the table above), so dropping it avoids imputing values for otherwise complete rows and keeps the baseline on the core features.

In [ ]:
decision_log = []


def log_step(decision, why, rows_before, rows_after):
    decision_log.append(
        {"decision": decision, "why": why, "rows_before": rows_before, "rows_after": rows_after}
    )


df = df_raw.drop(columns=DROP_COLS)
assert len(df) == len(df_raw)  # dropping columns never removes rows

log_step("Drop `area_type`", "Judgment call: changes what `total_sqft` measures; re-test in Session 5",
         len(df_raw), len(df))
log_step("Drop `availability`", "Possession label/date, not a property attribute", len(df_raw), len(df))
log_step("Drop `society`", "Large share missing and very high-cardinality", len(df_raw), len(df))
log_step("Drop `balcony`", "Minor amenity count; avoids imputing", len(df_raw), len(df))

df.shape

## 4. Remaining nulls in `location`, `size` and `bath`

Count them, look at where they sit, and check that they do not cluster before dropping them.

In [ ]:
CORE_NULL_COLS = ["location", "size", "bath"]

nulls_now = df.isnull().sum()
display(nulls_now.to_frame("nulls"))
assert set(nulls_now[nulls_now > 0].index) <= set(CORE_NULL_COLS), "unexpected nulls outside location/size/bath"

In [ ]:
null_flags = df[CORE_NULL_COLS].isnull()
null_mask = null_flags.any(axis=1)
n_null_rows = int(null_mask.sum())

print(f"Rows with at least one null in {CORE_NULL_COLS}: {n_null_rows} "
      f"({n_null_rows / len(df):.2%} of {len(df):,})")

# Which of the three columns are null together? (True = null)
null_flags[null_mask].value_counts().rename("rows").reset_index()

In [ ]:
# Raw rows (dropped columns still visible) for the null `location` and `size` cases.
df_raw[df["location"].isnull() | df["size"].isnull()]

In [ ]:
# A sample of the rows where only `bath` is null.
df_raw[df["bath"].isnull() & df["location"].notnull() & df["size"].notnull()].head(10)

### Do the null rows cluster?

Three views: by `location` (the check the brief asks for, benchmarked against random scatter), by `area_type` (read from the raw frame, because the column is already gone from `df`) and by position in the file.

In [ ]:
loc_all = df["location"].value_counts()                    # listings per location, all rows
loc_null = df.loc[null_mask, "location"].value_counts()    # null rows per location (null locations excluded)
n_known_null = int(loc_null.sum())

by_location = pd.DataFrame({"null_rows": loc_null, "all_rows": loc_all.reindex(loc_null.index)})
by_location["null_rate_%"] = (by_location["null_rows"] / by_location["all_rows"] * 100).round(1)
# Null rows a location would get if the nulls were scattered at random over all listings;
# lift = actual / expected (about 1 means "as many as its size predicts").
exp_loc = n_known_null * by_location["all_rows"] / loc_all.sum()
by_location["expected_if_random"] = exp_loc.round(1)
by_location["lift"] = (by_location["null_rows"] / exp_loc).round(1)
by_location = by_location.sort_values("null_rows", ascending=False)

print(f"{n_known_null} null rows have a known location, spread over {len(loc_null)} distinct "
      f"locations (of {len(loc_all):,} locations in the data).")
by_location.head(10)

In [ ]:
# Is the busiest location busier than random scatter would make it?
# Place the same number of null rows at random over all rows with a known location, 2,000 times,
# and record how many land in the single busiest location each time.
if by_location.empty:
    print("No null row has a known location.")
else:
    codes, _ = pd.factorize(df["location"].dropna())
    rng = np.random.default_rng(0)
    sim_max = np.array([
        np.bincount(codes[rng.choice(len(codes), size=n_known_null, replace=False)]).max()
        for _ in range(2000)
    ])
    observed_max = int(by_location["null_rows"].iloc[0])
    p99 = int(np.percentile(sim_max, 99))
    print(f"Observed: {observed_max} null rows in the busiest location ({by_location.index[0]!r}).")
    print(f"Random scatter: median {int(np.median(sim_max))}, 99th percentile {p99}; "
          f"{(sim_max >= observed_max).mean():.1%} of trials reach {observed_max} or more.")
    if observed_max > p99:
        print("CLUSTER WARNING: the busiest location holds more null rows than random scatter explains.")
    else:
        print("No sign of clustering by location: the busiest location is within what random scatter produces.")

In [ ]:
# area_type is already dropped from df, but the raw frame still has it: diagnostic use only.
area_check = pd.DataFrame({
    "null_rows": df_raw.loc[null_mask, "area_type"].value_counts(),
    "all_rows": df_raw["area_type"].value_counts(),
}).fillna(0).astype(int)
area_check["null_rate_%"] = (area_check["null_rows"] / area_check["all_rows"] * 100).round(2)
exp_area = n_null_rows * area_check["all_rows"] / area_check["all_rows"].sum()  # expected if scattered at random
area_check["expected_if_random"] = exp_area.round(1)
area_check["lift"] = (area_check["null_rows"] / exp_area).round(2)
area_check.sort_values("null_rows", ascending=False)

In [ ]:
# Where do the null rows sit in the file? Count them per tenth of the file (top -> bottom).
positions = np.flatnonzero(null_mask.to_numpy())
per_tenth = np.bincount(positions * 10 // len(df), minlength=10)
pd.DataFrame(
    [per_tenth],
    index=["null_rows"],
    columns=[f"{i * 10}-{(i + 1) * 10}%" for i in range(10)],
)

**Decision: drop these rows rather than impute them.** `location` has no sensible fill value, and `bath` is structurally tied to the bedroom count in `size`, so a median fill would invent data. Dropping is only safe when the affected share is small and spread out; the checks above report both.

In [ ]:
rows_before = len(df)
df = df.dropna()  # the index is deliberately not reset: it keeps each row traceable to its line in the raw CSV
rows_after = len(df)
assert rows_before - rows_after == n_null_rows  # exactly the rows flagged above

log_step(
    "Drop rows with a null in `location`, `size` or `bath`",
    "`location` has no sensible fill value; `bath` is tied to bedroom count (`size`), so imputing would invent data",
    rows_before, rows_after,
)

print(f"Rows before: {rows_before:,} | after: {rows_after:,} | removed: {rows_before - rows_after:,} "
      f"({(rows_before - rows_after) / rows_before:.2%})")

## 5. Final check

In [ ]:
assert list(df.columns) == ["location", "size", "total_sqft", "bath", "price"]
assert df.isnull().sum().sum() == 0

print(f"Final shape: {df.shape}")

null_summary = pd.DataFrame({"nulls_before": df_raw.isnull().sum()})
null_summary["nulls_after"] = df.isnull().sum()  # aligns on column name; dropped columns become missing
null_summary.astype("Int64")

`<NA>` in `nulls_after` marks a column that was dropped.

In [ ]:
df.head()

## 6. Decision log

Each decision with its row impact, generated from the counts recorded while the notebook ran.

In [ ]:
rows_raw = decision_log[0]["rows_before"]
rows_final = decision_log[-1]["rows_after"]
removed_total = rows_raw - rows_final

lines = [
    "| # | Decision | Why | Rows before | Rows after | Rows removed |",
    "|--:|----------|-----|------------:|-----------:|-------------:|",
]
for i, step in enumerate(decision_log, start=1):
    removed = step["rows_before"] - step["rows_after"]
    lines.append(
        f"| {i} | {step['decision']} | {step['why']} | "
        f"{step['rows_before']:,} | {step['rows_after']:,} | {removed:,} |"
    )

cols = ", ".join(f"`{c}`" for c in df.columns)
lines += [
    "",
    f"**Result:** {rows_raw:,} raw rows -> {rows_final:,} rows "
    f"({removed_total:,} removed, {removed_total / rows_raw:.2%} of the raw data; the brief expected under 1%), "
    f"{df.shape[1]} columns ({cols}), zero nulls. Prices are still in lakhs of INR.",
]
display(Markdown("\n".join(lines)))